# 🤖 OpenAI 프로그래밍 실습 노트북

오늘은 Jupyter Notebook에서 다음 내용을 실습합니다.
- 🎯 [0. OpenAI 키/엔드포인트 설정](#-0-사전-준비0.-사전-준비)
- 🎓 [실습 1: Chat Completion](#-실습-1-chat-completion)
- 🎓 [실습 2: 프롬프트 기반 텍스트 생성](#-실습-2-프롬프트-기반-텍스트-생성)
- 🎓 [실습 3: 이미지 분석](#-실습-3-이미지-분석)
- ✏️ [미니 과제 1: 간단한 질의응답(FAQ) 봇 만들기](#️-미니-과제-1-간단한-질의응답faq-봇)
- 🎓 [실습 4: 도구(Tool) 정의하기](#-실습-4-도구tool-정의하기-)
- ✏️ [미니 과제: 나만의 도구 정의하기](#️-미니-과제-나만의-도구-정의하기)
- 🎓 [실습 5: 벡터 임베딩](#-실습-5-벡터-임베딩)


## 🎓 0. 사전 준비
1. OpenAI 리소스 및 배포(Deployment) 생성
2. 배포 이름 확인 (예: `gpt-5.4`)
3. API Key, Endpoint 준비

### 🎯 1. OpenAI SDK 설치
OpenAI 모델을 사용하기 위해 OpenAI SDK를 설치합니다.

- DevContainer를 이용하는 경우

In [5]:
!uv pip install -r ../pyproject.toml
!uv sync

Using Python 3.12.13 environment at: /home/kymun/krmunio/maf-ai-class-2026/.venv
Resolved 170 packages in 69ms                                        
Checked 170 packages in 1ms
Resolved 177 packages in 0.88ms
Checked 170 packages in 1ms


- GitHub Codespaces를 이용하는 경우

In [6]:
!pip install .

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

### 🎯 2. OpenAI 환경변수 설정
- 가능한 경우 OS 환경변수를 사용하세요.
- 비어 있으면 노트북에서 안전하게 입력받습니다.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # .env 파일 로드

API_KEY = os.getenv("API_KEY")
ENDPOINT = os.getenv("ENDPOINT")
CHAT_DEPLOYMENT = os.getenv("CHAT_DEPLOYMENT")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT")
EMBEDDING_ENDPOINT = os.getenv("EMBEDDING_ENDPOINT")
EMBEDDING_DEPLOYMENT = os.getenv("EMBEDDING_DEPLOYMENT")

print("- OPENAI_API_KEY:", API_KEY)
print("- Endpoint:", ENDPOINT)
print("- Chat deployment:", CHAT_DEPLOYMENT)
print("- Vision deployment:", VISION_DEPLOYMENT)
print("- Embedding endpoint:", EMBEDDING_ENDPOINT)
print("- Embedding deployment:", EMBEDDING_DEPLOYMENT)


- OPENAI_API_KEY: 56af3dd4b8c144d9b9148e4920d38c7b
- Endpoint: https://apim-ai-aaf-010.azure-api.net
- Chat deployment: gpt-4.1-mini
- Vision deployment: gpt-4.1-mini
- Embedding endpoint: https://apim-ai-aaf-010.azure-api.net
- Embedding deployment: text-embedding-3-large


---

## 🎓 실습 1: Chat Completion
간단한 대화형 응답을 받아봅니다.


In [8]:
import os
from openai import AzureOpenAI

api_key = os.environ.get("API_KEY")
client = AzureOpenAI(
    api_key = api_key,
    azure_endpoint = os.environ.get("ENDPOINT", ""),
    api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    default_headers = {
        "Ocp-Apim-Subscription-Key": api_key,
    },
)

CHAT_MODEL = os.environ.get("CHAT_DEPLOYMENT", "")
VISION_MODEL = os.environ.get("VISION_DEPLOYMENT", "")

response = client.chat.completions.create(
    model = CHAT_MODEL,
    messages = [{"role": "user", "content": "Reply with the single word READY."}],
    max_tokens = 64,
)

print(response.choices[0].message.content)

READY


## 🎓 실습 2: 프롬프트 기반 텍스트 생성
프롬프트를 바꿔가며 스타일/톤 변화를 확인합니다.


In [9]:
prompt = "청소년 AI 워크숍 홍보 문구를 5문장으로 작성해줘. 톤은 밝고 쉬운 한국어로."

response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "너는 마케팅 카피라이터다."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.9,
)

print(response.choices[0].message.content)

미래를 만드는 AI, 지금 바로 만나보세요! 청소년 AI 워크숍에서 재미있게 배우고 직접 체험할 수 있어요. 어렵지 않아요, 누구나 쉽게 따라올 수 있답니다. 친구들과 함께 참여하면 더 신나고 즐거워요! 지금 신청해서 나만의 AI 세상을 만들어 보세요!


## 🎓 실습 3: 이미지 분석
이미지 URL을 입력하여 내용 요약을 요청합니다.


In [10]:
image_url = "https://imgnews.pstatic.net/image/029/2026/04/14/0003021534_002_20260414102812207.jpg"

response = client.chat.completions.create(
    model=VISION_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "이 이미지를 한국어로 5문장 이내로 설명해줘."},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
)

print(response.choices[0].message.content)

이 사진은 KB데이터시스템과 Microsoft가 함께하는 AI 인재 양성 프로젝트 행사입니다. 많은 청년들이 모여 프로젝트를 기념하며 단체 사진을 찍고 있습니다. 참가자들은 '유스 AI 프로젝트:D'라는 배너를 들고 있습니다. 행사장은 밝고 현대적인 강의실 분위기입니다. 모두가 밝은 표정으로 활기차게 참여하는 모습입니다.


## ✏️ 미니 과제 1: 간단한 질의응답(FAQ) 봇

작은 지식 베이스를 기반으로 답하는 함수형 봇 예시입니다.


In [11]:
faq_context = """
[워크숍 정보]
- 대상: 중학생~고등학생
- 준비물: 개인 노트북(크롬 설치), Microsoft 계정
- 실습 내용: 챗봇 만들기, 이미지 이해, 프롬프트 엔지니어링 기초
- 시간: 총 2시간
"""

def ask_faq_bot(question, history=None):
    if history is None:
        history = []

    messages = [
        {
            "role": "system",
            "content": (
                "너는 워크숍 안내 봇이다. 제공된 정보 범위에서만 답하고,"
                "정보가 없으면 없다고 말해라."
            ),
        },
        {"role": "system", "content": faq_context},
    ] + history + [{"role": "user", "content": question}]

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
    )

    answer = response.choices[0].message.content
    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return answer, history

chat_history = []
for q in ["준비물이 뭐야?", "몇 시간 진행돼?", "점심 제공돼?"]:
    a, chat_history = ask_faq_bot(q, chat_history)
    print(f"Q: {q}")
    print(f"A: {a}\n")

Q: 준비물이 뭐야?
A: 준비물은 개인 노트북(크롬 설치)와 Microsoft 계정입니다.

Q: 몇 시간 진행돼?
A: 워크숍은 총 2시간 동안 진행됩니다.

Q: 점심 제공돼?
A: 점심 제공에 대한 정보는 없습니다.



---

## 🎓 실습 4: 도구(Tool) 정의하기 🔧

### 1. 도구(Tool) 정의 (JSON Schema)

도구는 **모델이 직접 실행하는 함수가 아니라**,
모델이 *호출 요청(tool_calls)* 을 생성할 수 있도록 주는 **함수 인터페이스(설명서)** 입니다.

아래 예제는 `add_numbers(a, b)` 라는 간단한 도구를 정의합니다.

In [12]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'add_numbers',
            'description': '두 개의 숫자를 더해 합계를 반환합니다.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'a': {'type': 'number', 'description': '첫 번째 숫자'},
                    'b': {'type': 'number', 'description': '두 번째 숫자'}
                },
                'required': ['a', 'b'],
                'additionalProperties': False
            },
            # strict=True 를 주면, 모델이 schema를 엄격하게 따르도록 유도합니다.
            'strict': True
        }
    }
]
tools

[{'type': 'function',
  'function': {'name': 'add_numbers',
   'description': '두 개의 숫자를 더해 합계를 반환합니다.',
   'parameters': {'type': 'object',
    'properties': {'a': {'type': 'number', 'description': '첫 번째 숫자'},
     'b': {'type': 'number', 'description': '두 번째 숫자'}},
    'required': ['a', 'b'],
    'additionalProperties': False},
   'strict': True}}]

### 2. 실제로 실행될 함수 구현

**중요:** 모델은 함수를 실행하지 않습니다. 실행은 **여기(로컬 코드)** 에서 합니다.

In [13]:
def add_numbers(a: float, b: float) -> float:
    return a + b

add_numbers(1, 2)

3

### 3. 모델 호출 → tool_call 받기 (1차 요청)

모델에게 tools 목록을 주고 질문합니다. 질문에 따라 모델이 tool_calls를 반환할 수 있습니다.

In [14]:
import json

messages = [
    {'role': 'system', 'content': '당신은 필요한 경우 도구를 호출해 정확한 답을 만드는 도우미입니다.'},
    {'role': 'user', 'content': '12.5와 7.3을 더해서 결과를 알려줘.'}
]

resp = client.chat.completions.create(
    model = CHAT_DEPLOYMENT,
    messages = messages,
    tools = tools,
    tool_choice = 'auto',
)

msg = resp.choices[0].message

### 4. tool_calls 파싱 후 로컬 함수 실행

모델이 `tool_calls` 를 반환했다면,
각 tool_call의 `function.name` 과 `function.arguments(JSON string)` 를 읽어
해당 Python 함수를 실행하고 결과를 `tool` 역할 메시지로 추가합니다.

In [15]:
tool_calls = msg.tool_calls or []
print('tool_calls count =', len(tool_calls))

for call in tool_calls:
    fn_name = call.function.name
    args = json.loads(call.function.arguments)

    if fn_name == 'add_numbers':
        result = add_numbers(args['a'], args['b'])
        # tool 결과를 tool 메시지로 추가
        messages.append({
            'role': 'assistant',
            'content': None,
            'tool_calls': [call],
        })
        messages.append({
            'role': 'tool',
            'tool_call_id': call.id,
            'content': json.dumps({'result': result}, ensure_ascii=False)
        })
    else:
        raise ValueError(f'Unknown tool: {fn_name}')

messages[-2:]

tool_calls count = 1


[{'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_HRtX3wntE5NYzDS9HEUy3TVj', function=Function(arguments='{"a":12.5,"b":7.3}', name='add_numbers'), type='function')]},
 {'role': 'tool',
  'tool_call_id': 'call_HRtX3wntE5NYzDS9HEUy3TVj',
  'content': '{"result": 19.8}'}]

### 5. tool 결과를 포함해 최종 응답 생성 (2차 요청)

이제 모델은 tool 실행 결과를 보고 자연어로 최종 답변을 만들어 줍니다.

In [16]:
final = client.chat.completions.create(
    model = CHAT_DEPLOYMENT,
    messages = messages,
    tools = tools,
)

print(final.choices[0].message.content)

12.5와 7.3을 더한 결과는 19.8입니다.


## ✏️ 미니 과제: 나만의 도구 추가하기

아래 아이디어 중 하나를 골라 새로운 도구를 추가해보세요!

1. **단위 변환기**: km↔mile, kg↔lb, ℃↔℉
2. **주사위 굴리기**: 랜덤 숫자 생성
3. **날짜 계산기**: D-day 계산
4. **진로 추천**: 청소년에게 진로 추천하기

In [17]:
# 👇 여기에 새로운 도구를 만들어보세요!

def my_new_tool(param1: str) -> str:
    """새 도구 설명"""
    # 여기에 구현
    return "결과"

# tools 리스트에 새 도구 스펙 추가
# available_tools 딕셔너리에 매핑 추가
# run_agent()로 테스트!

---

## 🎓 실습 5: 벡터 임베딩

임베딩(Embedding)의 개념 이해하고, 문장을 벡터로 변환합니다.

### 1. 임베딩 생성 실습

In [18]:
texts = [
    "고양이는 귀여운 동물이다",
    "강아지는 사람에게 친근하다",
    "자동차는 교통수단이다",
    "AI는 데이터를 이해한다"
]

embedding_client = AzureOpenAI(
    api_key = API_KEY,
    azure_endpoint = os.environ.get("EMBEDDING_ENDPOINT", os.environ.get("ENDPOINT", "")),
    api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    default_headers = {
        "Ocp-Apim-Subscription-Key": API_KEY,
    },
)

response = embedding_client.embeddings.create(
    model = EMBEDDING_DEPLOYMENT,
    input = texts,
)

embeddings = [item.embedding for item in response.data]

print(f"벡터 차원: {len(embeddings[0])}")


벡터 차원: 3072


### 2. 코사인 유사도 계산

- 0.8 이상 : 의미적으로 매우 유사
- 0.5~0.7 : 어느 정도 유사

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_matrix = cosine_similarity(np.array(embeddings))

for i in range(len(texts)):
    for j in range(len(texts)):
        print(f"{i}-{j}: {similarity_matrix[i][j]:.3f}")
    print()

0-0: 1.000
0-1: 0.486
0-2: 0.221
0-3: 0.233

1-0: 0.486
1-1: 1.000
1-2: 0.166
1-3: 0.314

2-0: 0.221
2-1: 0.166
2-2: 1.000
2-3: 0.273

3-0: 0.233
3-1: 0.314
3-2: 0.273
3-3: 1.000



In [20]:
query = "사람과 친한 동물"
query_embedding = embedding_client.embeddings.create(
    model = EMBEDDING_DEPLOYMENT,
    input = query
).data[0].embedding

scores = cosine_similarity(
    [query_embedding],
    embeddings
)[0]

for text, score in sorted(zip(texts, scores), key=lambda x: x[1], reverse=True):
    print(f"{score:.3f} | {text}")


0.593 | 강아지는 사람에게 친근하다
0.441 | 고양이는 귀여운 동물이다
0.302 | AI는 데이터를 이해한다
0.188 | 자동차는 교통수단이다


---
## 🎉 수고하셨습니다!

오늘 배운 내용:
- **모델 호출**: 모델을 호출하고 프롬프트를 기반으로 Chat Completion
- **도구 호출**: AI가 도구를 선택하고 호출하는 방법
- **벡터 임베딩**: 임베딩의 개념 이해하고 텍스트를 임베딩으로 변환

### 📝 숙제: 임베딩을 도구로 바꿔보기
